# Whisper v2 与 v3 同 50 条对比

In [1]:
import csv
import gc
import re
import time
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import torch
from IPython.display import display

from peft import PeftModel
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
)

### 定义相关参数

In [20]:
LANG = "fr"  # fr / de / nn

WHISPER_LANGUAGE = {
    "fr": "french",
    "de": "german",
    "nn": "nynorsk",
}[LANG]

In [21]:
# datasets 路径
CV_ROOT = Path(
    "/home/sigan/datasets/common_voice_27_fr/cv-corpus-27.0-2026-09-11/fr"
)

NUM_SAMPLES = 50
SAMPLING_RATE = 16000
MAX_NEW_TOKENS = 100
NUM_BEAMS = 1
DEVICE = "cuda"

# 原始 v2/v3 对比时保持 None。
# v2 微调完成后，填入 adapter 目录即可。
V2_ADAPTER_DIR = "/sigan/ai-ml/models/whisper-large-v2-fr-lora"
# V2_ADAPTER_DIR = None
# v2 基座模型
V2_MODEL_ID = "openai/whisper-large-v2"

# 如果以后也训练了 v3，可填写 v3 adapter 路径。
V3_ADAPTER_DIR = "/sigan/ai-ml/models/whisper-large-v3-fr-lora"
# V3_ADAPTER_DIR = None
# v3 基座模型
V3_MODEL_ID = "openai/whisper-large-v3"

MODEL_SPECS = [
    {
        "name": "Whisper-v2",
        "model_id": V2_MODEL_ID,
        "adapter_dir": V2_ADAPTER_DIR,
    },
    {
        "name": "Whisper-v3",
        "model_id": V3_MODEL_ID,
        "adapter_dir": None
    },
]

### 读取处理数据

#### 定义归一函数

In [22]:
import re
import unicodedata

def normalize_text(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [23]:
# 读取相同 50 条测试数据
with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    rows = list(
        csv.DictReader(handle, delimiter="\t")
    )[:NUM_SAMPLES]

references = []
audio_arrays = []

# 音频只读取一次，两个模型使用完全相同的波形
for index, row in enumerate(rows, start=1):
    audio_path = CV_ROOT / "clips" / row["path"]

    audio, _ = librosa.load(
        str(audio_path),
        sr=SAMPLING_RATE,
    )

    # references.append(row["sentence"])
    # 标准答案也使用同一套归一化规则
    references.append(
        normalize_text(row["sentence"])
    )
    audio_arrays.append(audio)

    if index % 10 == 0:
        print(f"音频已加载：{index}/{NUM_SAMPLES}")

# 使用更完善的归一函数
# def clean_text(text):
#     # 去掉不同模型可能插入的空格
#     return re.sub(r"\s+", "", text).strip()


def run_model(model_id, adapter_dir):
    # v2 使用 80 Mel，v3 使用 128 Mel，所以每个模型加载自己的 Processor
    processor = AutoProcessor.from_pretrained(
        model_id,
        language=WHISPER_LANGUAGE,
        task="transcribe",
    )
    
    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=WHISPER_LANGUAGE,
        task="transcribe",
    )

    # 8-bit 加载，确保两个模型可以分别在 16GB 显存上运行
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        quantization_config=BitsAndBytesConfig(
            load_in_8bit=True
        ),
        device_map="auto",
    )

    base_model.config.use_cache = True
    model = base_model

    # 可选：挂载训练后的 LoRA adapter
    if adapter_dir:
        model = PeftModel.from_pretrained(
            base_model,
            adapter_dir,
        )

    model.eval()

    # 不要重复传 forced_decoder_ids，否则可能出现参数冲突
    model.generation_config.forced_decoder_ids = (
        forced_decoder_ids
    )

    predictions = []
    start_time = time.perf_counter()

    for index, audio in enumerate(audio_arrays, start=1):
        inputs = processor(
            audio,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
        ).to(DEVICE)

        # 8-bit 模型的卷积层使用 FP16
        inputs["input_features"] = (
            inputs["input_features"].half()
        )

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
            )

        prediction = processor.batch_decode(
            output_ids,
            skip_special_tokens=True,
        )[0]

        predictions.append(
            normalize_text(prediction)
        )

        if index % 5 == 0:
            print(f"已生成：{index}/{NUM_SAMPLES}")

    elapsed = time.perf_counter() - start_time

    # 及时释放显存，准备加载下一个模型
    del model
    del base_model
    del processor
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, elapsed

音频已加载：10/50
音频已加载：20/50
音频已加载：30/50
音频已加载：40/50
音频已加载：50/50


### 开始测试数据

In [24]:
all_results = {}

In [25]:
for spec in MODEL_SPECS:
    print("\n开始测试：", spec["name"])

    predictions, elapsed = run_model(
        spec["model_id"],
        spec["adapter_dir"],
    )

    overall_wer = jiwer.wer(
        reference=references,
        hypothesis=predictions,
    )
    
    all_results[spec["name"]] = {
        "predictions": predictions,
        "wer": overall_wer,
        "runtime": elapsed,
    }

    print(
        spec["name"],
        "总体 WER：",
        overall_wer,
        "耗时：",
        round(elapsed, 2),
        "秒",
    )


开始测试： Whisper-v2


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


已生成：5/50
已生成：10/50
已生成：15/50
已生成：20/50
已生成：25/50
已生成：30/50
已生成：35/50
已生成：40/50
已生成：45/50
已生成：50/50
Whisper-v2 总体 WER： 0.08610567514677103 耗时： 109.06 秒

开始测试： Whisper-v3


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


已生成：5/50
已生成：10/50
已生成：15/50
已生成：20/50
已生成：25/50
已生成：30/50
已生成：35/50
已生成：40/50
已生成：45/50
已生成：50/50
Whisper-v3 总体 WER： 0.0821917808219178 耗时： 93.22 秒


### 对比数据

In [26]:
# 生成逐条对比表
v2 = all_results["Whisper-v2"]["predictions"]
v3 = all_results["Whisper-v3"]["predictions"]

table_rows = []

for index, reference in enumerate(references, start=1):
    v2_wer = jiwer.wer(
        reference=reference,
        hypothesis=v2[index - 1],
    )
    
    v3_wer = jiwer.wer(
        reference=reference,
        hypothesis=v3[index - 1],
    )
    
    table_rows.append({
        "序号": index,
        "标准答案": reference,
        "v2 输出": v2[index - 1],
        "v3 输出": v3[index - 1],
        "v2 WER": round(v2_wer, 4),
        "v3 WER": round(v3_wer, 4),
        "v3改善": round(v2_wer - v3_wer, 4),
    })

comparison_df = pd.DataFrame(table_rows)
comparison_df = comparison_df.sort_values(
    by="v3改善",
    ascending=False,
)

display(comparison_df)

v3_better = (comparison_df["v3改善"] > 0).sum()
equal_count = (comparison_df["v3改善"] == 0).sum()
v3_worse = (comparison_df["v3改善"] < 0).sum()

,序号,标准答案,v2 输出,v3 输出,v2 WER,v3 WER,v3改善
34,35,berthe hess,bert s,berthe s,1.0000,0.5000,0.5000
47,48,en gras disciplines à examens,angra discipline à examen,en gras discipline à examen,0.8000,0.4000,0.4000
29,30,beaucoup de ses ouvrages se trouvent dans le f...,beaucoup de ses ouvrages se trouvent dans le f...,beaucoup de ces ouvrages se trouvent dans le f...,0.3077,0.1538,0.1538
13,14,le drapé des vêtements est parfaitement réalisé,le draper des vêtements est parfaitement réalisé,le drapé des vêtements est parfaitement réalisé,0.1429,0.0000,0.1429
9,10,en juin participent au boycott des wagons pullman,en juin participe au boycott des wagons pullman,en juin participent au boycott des wagons pullman,0.1250,0.0000,0.1250
11,12,son frère drew hawkins joue pour boise state,son frère drew watkins joue pour boyd state,son frère drew hawkins joue pour boyd state,0.2500,0.1250,0.1250
5,6,la délicatesse et la douceur des coloris évoqu...,la délicatesse et la douceur des coulories évo...,la délicatesse et la douceur des coloris évoqu...,0.1667,0.0833,0.0833
38,39,il me semble que cela s applique particulièrem...,il me semble que cela s applique particulièrem...,il me semble que cela s applique particulièrem...,0.0000,0.0000,0.0000
30,31,viollet leduc édition hugues mille huit cent s...,violer le duc édition uq 1877,violez le duc édition uq 1877,1.0000,1.0000,0.0000
31,32,il est le frère d emidio graça aussi footballe...,il est le frère d emilio graça aussi footballe...,il est le frère d emilio graça aussi footballe...,0.0909,0.0909,0.0000


### 总结

In [27]:
print("\n总体对比")
print("v2 WER：", all_results["Whisper-v2"]["wer"])
print("v3 WER：", all_results["Whisper-v3"]["wer"])
print("v3 更好：", v3_better)
print("两者相同：", equal_count)
print("v3 更差：", v3_worse)


总体对比
v2 WER： 0.08610567514677103
v3 WER： 0.0821917808219178
v3 更好： 7
两者相同： 40
v3 更差： 3
